In [6]:
# Pretraining + Recursive Forecasting for Insulin Scheduling
# Colab/Jupyter-ready. Requires: pandas, numpy, sklearn, xgboost

from pathlib import Path
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from copy import deepcopy

# ---------- 0. Load dataset ----------
default_path = Path("/mnt/data/insulin_dataset.csv")
alt_path = Path("insulin_dataset.csv")

def load_dataset():
    if default_path.exists():
        print(f"Loading dataset from {default_path}")
        return pd.read_csv(default_path)
    if alt_path.exists():
        print(f"Loading dataset from {alt_path.resolve()}")
        return pd.read_csv(alt_path)
    # Try Google Colab upload
    try:
        from google.colab import files
        print("No local file found. Please upload 'insulin_dataset.csv'.")
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        return pd.read_csv(fname)
    except Exception:
        raise FileNotFoundError(
            "Could not find 'insulin_dataset.csv'. Place it in the notebook folder or run in Colab and upload when prompted."
        )

df_raw = load_dataset()
print("Loaded dataset. Top rows:")
display(df_raw.head())

# ---------- 1. Preprocess ----------
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

# Find a timestamp-like column
ts_candidates = [c for c in df.columns if "timestamp" in c.lower() or "time" in c.lower()]
if not ts_candidates:
    raise ValueError("No timestamp-like column found. Ensure your CSV has a timestamp column.")
ts_col = ts_candidates[0]
print("Using timestamp column:", ts_col)

# Parse timestamps; prefer day-first because your sample used day-first
df[ts_col] = pd.to_datetime(df[ts_col], dayfirst=True, errors='coerce')
if df[ts_col].isna().any():
    print("Warning: some timestamps failed to parse. Inspect your timestamp format.")

df = df.sort_values(by=ts_col).reset_index(drop=True)
df["hour"] = df[ts_col].dt.hour.fillna(0).astype(int)
df["dayofweek"] = df[ts_col].dt.dayofweek.fillna(0).astype(int)
df["time_gap_hours"] = df[ts_col].diff().dt.total_seconds().div(3600).fillna(np.nan)
# Default first gap to median or 24
df.loc[df.index[0], "time_gap_hours"] = df["time_gap_hours"].median() if df["time_gap_hours"].notna().any() else 24.0
df["time_gap_hours"] = df["time_gap_hours"].fillna(df["time_gap_hours"].median())

# Heuristic column detection for patient features
def find_col(keywords):
    for c in df.columns:
        lc = c.lower()
        if all(k in lc for k in keywords):
            return c
    return None

col_age = find_col(["age"]) or "Age" if "Age" in df.columns else None
col_bmi = find_col(["bmi"]) or "BMI" if "BMI" in df.columns else None
col_glucose = find_col(["glucose"]) or "Glucose" if "Glucose" in df.columns else None
col_basal = find_col(["basal"]) or "Basal Insulin (Units)" if "Basal Insulin (Units)" in df.columns else None
col_meal = find_col(["meal", "bolus"]) or "Meal Bolus (Units)" if "Meal Bolus (Units)" in df.columns else None
col_corr = find_col(["correction"]) or "Correction Bolus (Units)" if "Correction Bolus (Units)" in df.columns else None
col_total = find_col(["total", "bolus"]) or "Total Bolus (Units)" if "Total Bolus (Units)" in df.columns else None
col_type = None
for c in df.columns:
    if "predicted" in c.lower() and "type" in c.lower():
        col_type = c
        break
if col_type is None:
    for c in df.columns:
        if "type" == c.lower() or "diabetes" in c.lower():
            col_type = c
            break

# Provide fallbacks where missing
if col_age not in df.columns:
    df["age"] = df.get("Age", 40)
    col_age = "age"
if col_bmi not in df.columns:
    df["bmi"] = df.get("BMI", 25)
    col_bmi = "bmi"
if col_glucose not in df.columns:
    df["glucose"] = df.get("Glucose", 120)
    col_glucose = "glucose"
if col_basal not in df.columns:
    df["basal"] = df.get("Basal Insulin (Units)", 0).fillna(0)
    col_basal = "basal"
if col_meal not in df.columns:
    df["meal"] = df.get("Meal Bolus (Units)", 0).fillna(0)
    col_meal = "meal"
if col_corr not in df.columns:
    df["correction"] = df.get("Correction Bolus (Units)", 0).fillna(0)
    col_corr = "correction"
if col_total not in df.columns:
    # build total from meal + correction if not present
    df["total"] = df[col_meal].fillna(0) + df[col_corr].fillna(0)
    col_total = "total"
if col_type not in df.columns:
    df["predicted_type"] = df.get("Predicted Type", "Unknown")
    col_type = "predicted_type"

# Encode diabetes type
le = LabelEncoder()
df[col_type] = df[col_type].fillna("Unknown").astype(str)
df["pred_type_enc"] = le.fit_transform(df[col_type])

# Final features
feature_cols = [
    col_age, col_bmi, col_glucose, "pred_type_enc",
    col_basal, col_meal, col_corr, col_total,
    "hour", "dayofweek"
]

# Ensure numeric
for f in feature_cols:
    df[f] = pd.to_numeric(df[f], errors='coerce').fillna(0.0)

# Targets
df["target_total_bolus"] = pd.to_numeric(df[col_total], errors='coerce').fillna(0.0)
df["target_time_gap"] = pd.to_numeric(df["time_gap_hours"], errors='coerce').fillna(24.0)

print("Prepared feature sample:")
display(df[feature_cols + ["target_total_bolus","target_time_gap"]].head())

# ---------- 2. Train/test split ----------
X = df[feature_cols].copy()
y_bolus = df["target_total_bolus"].copy()
y_gap = df["target_time_gap"].copy()

X_train, X_test, yb_train, yb_test, yg_train, yg_test = train_test_split(
    X, y_bolus, y_gap, test_size=0.2, random_state=42
)
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

# ---------- 3. Train XGBoost models ----------
model_bolus = XGBRegressor(objective="reg:squarederror", n_estimators=150, learning_rate=0.08, random_state=42)
model_gap = XGBRegressor(objective="reg:squarederror", n_estimators=150, learning_rate=0.08, random_state=42)

print("Training Total Bolus model...")
model_bolus.fit(X_train, yb_train)
print("Training Time Gap model...")
model_gap.fit(X_train, yg_train)

# Evaluate
yb_pred = model_bolus.predict(X_test)
yg_pred = model_gap.predict(X_test)
print(f"Bolus MAE: {mean_absolute_error(yb_test, yb_pred):.3f} units")
print(f"Time-gap MAE: {mean_absolute_error(yg_test, yg_pred):.3f} hours")

# ---------- 4. Recursive forecasting function ----------
def recursive_forecast(pretrained_bolus_model, pretrained_gap_model, user_input, days=7, verbose=False):
    """
    Recursive insulin bolus & time-gap forecast (hour/dayofweek fixed).

    user_input: dict containing all keys in feature_cols
    days: number of days to predict recursively
    Returns: DataFrame with daily predictions
    """
    missing = [k for k in feature_cols if k not in user_input]
    if missing:
        raise ValueError(f"Missing keys in user_input: {missing}")

    cur = deepcopy(user_input)
    preds = []

    for d in range(1, days + 1):
        Xcur = pd.DataFrame([cur])[feature_cols]

        # Predict bolus and time-gap
        bolus_pred = float(pretrained_bolus_model.predict(Xcur)[0])
        gap_pred = float(pretrained_gap_model.predict(Xcur)[0])

        preds.append({
            "day": d,
            "predicted_total_bolus": round(bolus_pred, 2),
            "predicted_time_gap_hours": round(gap_pred, 2),
            "input_hour": cur["hour"],
            "input_dayofweek": cur["dayofweek"]
        })

        if verbose:
            print(f"Day {d}: bolus={bolus_pred:.2f} units, gap={gap_pred:.2f} hrs")

        # --- Update current input ---
        cur[col_total] = bolus_pred
        cur["total"] = bolus_pred
        cur["pred_type_enc"] = int(cur.get("pred_type_enc", 0))

        # KEEP HOUR AND DAYOFWEEK FIXED for smooth weekly forecast
        # cur["hour"] = cur["hour"]
        # cur["dayofweek"] = cur["dayofweek"]

        # Other features (BMI, glucose, etc.) stay constant until re-anchor

    return pd.DataFrame(preds)



# ---------- 5. Demo using the dataset last row as Day-0 user input ----------
sample_row = X.iloc[-1].to_dict()
user_input = {k: float(sample_row[k]) for k in feature_cols}
user_input["pred_type_enc"] = int(user_input["pred_type_enc"])
user_input["hour"] = int(user_input["hour"])
user_input["dayofweek"] = int(user_input["dayofweek"])

print("Example Day-0 user input (from dataset):")
print(user_input)

print("\nRecursive forecast for 7 days:")
preds = recursive_forecast(model_bolus, model_gap, user_input, days=7, verbose=True)
display(preds)

# ---------- 6. Re-anchoring helper ----------
def reanchor_with_actuals(current_input, actuals_dict):
    """
    Replace fields in current_input with actuals provided at weekly check-in.
    actuals_dict keys are expected to match column names (like 'Total Bolus (Units)' or 'total' etc.)
    """
    next_input = deepcopy(current_input)
    for k in feature_cols:
        if k in actuals_dict:
            next_input[k] = actuals_dict[k]
    # if user supplies text diabetes type, encode it:
    if col_type in actuals_dict:
        try:
            next_input["pred_type_enc"] = int(le.transform([actuals_dict[col_type]])[0])
        except Exception:
            next_input["pred_type_enc"] = next_input.get("pred_type_enc", 0)
    return next_input

print("To re-anchor weekly, call reanchor_with_actuals(current_input, actuals_dict) with user-confirmed values.")

# Save models for reuse
try:
    model_bolus.save_model("model_bolus.json")
    model_gap.save_model("model_gap.json")
    print("Saved models to model_bolus.json and model_gap.json")
except Exception as e:
    print("Could not save models in this environment:", e)


Loading dataset from /content/insulin_dataset.csv
Loaded dataset. Top rows:


,Row,Age,BMI,Glucose,Predicted Type,Basal Insulin (Units),Meal Bolus (Units),Correction Bolus (Units),Total Bolus (Units),Timestamp (Generated),Notes
0,1,50,33.6,148,T2D,24,3.1,1.0,4.1,27-08-2025 08:15,NaN
1,2,31,23.3,85,Prediabetic,17,5.3,0.0,5.3,27-08-2025 14:30,NaN
2,3,32,28.1,183,T2D,20,3.8,1.7,5.5,28-08-2025 12:45,NaN
3,4,21,43.1,89,T2D,31,2.4,0.0,2.4,28-08-2025 19:00,NaN
4,5,33,25.6,137,T2D,19,3.9,0.7,4.6,29-08-2025 09:20,NaN


Using timestamp column: Timestamp (Generated)
Prepared feature sample:


,Age,BMI,Glucose,pred_type_enc,Basal Insulin (Units),Meal Bolus (Units),Correction Bolus (Units),Total Bolus (Units),hour,dayofweek,target_total_bolus,target_time_gap
0,50,33.6,148,2,24,3.1,1.0,4.1,8,2,4.1,0.750000
1,35,32.9,138,2,24,3.1,0.8,3.9,10,2,3.9,2.250000
2,54,27.4,171,2,20,3.8,1.4,5.2,10,2,5.2,0.083333
3,21,33.9,142,2,25,3.0,0.8,3.8,11,2,3.8,0.750000
4,44,41.5,146,2,30,2.5,0.9,3.4,12,2,3.4,1.000000


Train rows: 80, Test rows: 20
Training Total Bolus model...
Training Time Gap model...
Bolus MAE: 0.055 units
Time-gap MAE: 2.558 hours
Example Day-0 user input (from dataset):
{'Age': 32.0, 'BMI': 32.5, 'Glucose': 99.0, 'pred_type_enc': 2, 'Basal Insulin (Units)': 24.0, 'Meal Bolus (Units)': 3.1, 'Correction Bolus (Units)': 0.0, 'Total Bolus (Units)': 3.1, 'hour': 20, 'dayofweek': 1}

Recursive forecast for 7 days:
Day 1: bolus=3.10 units, gap=0.68 hrs
Day 2: bolus=3.10 units, gap=0.68 hrs
Day 3: bolus=3.10 units, gap=0.68 hrs
Day 4: bolus=3.10 units, gap=0.68 hrs
Day 5: bolus=3.10 units, gap=0.68 hrs
Day 6: bolus=3.10 units, gap=0.68 hrs
Day 7: bolus=3.10 units, gap=0.68 hrs


,day,predicted_total_bolus,predicted_time_gap_hours,input_hour,input_dayofweek
0,1,3.1,0.68,20,1
1,2,3.1,0.68,20,1
2,3,3.1,0.68,20,1
3,4,3.1,0.68,20,1
4,5,3.1,0.68,20,1
5,6,3.1,0.68,20,1
6,7,3.1,0.68,20,1


To re-anchor weekly, call reanchor_with_actuals(current_input, actuals_dict) with user-confirmed values.
Saved models to model_bolus.json and model_gap.json
